# Prompt Builder Demo - Yoga Pose Generation

## System Overview

This notebook demonstrates the prompt builder system:
- **YAML Wildcards** - Structured data (character, poses, photographic styles)
- **Jinja2 Templates** - Pluggable prompt templates with dynamic variable expansion
- **Prompt Builder** - Constructor pattern for composing image generation prompts

We'll generate 4 Thunderbolt Pose (Vajrasana) variations ready for ComfyUI/NanoBanana.

## Setup & Dependencies

**Required Packages Added to requirements.txt:**
- `pyyaml>=6.0` - YAML wildcard file loading
- `jinja2>=3.0` - Template rendering engine

**Docker Container Update:**
```bash
# Rebuild the Jupyter container with new dependencies
docker-compose build jupyter

# Restart the container
docker-compose up -d jupyter
```

The Dockerfile automatically installs all packages during image build. No manual pip install needed.

In [ ]:
import sys
from pathlib import Path

# Add prompt builder to path
sys.path.insert(0, str(Path.cwd() / 'data' / 'Prompts'))

from prompt_builder import PromptBuilder, Prompt
import json

print("✓ Prompt builder imported")

## Setup: Initialize the Builder

The builder needs to know where your wildcard YAML files are located.

In [ ]:
# Initialize builder pointing to wildcard YAML directory
wildcards_dir = Path.cwd() / 'data' / 'Prompts' / 'wildcards_yaml'
builder = PromptBuilder(wildcards_dir=str(wildcards_dir))

print(f"✓ Builder initialized")
print(f"  Wildcards directory: {wildcards_dir}")
print(f"  YAML files available:")
for f in sorted(wildcards_dir.glob("*.yaml")):
    print(f"    - {f.name}")

## Define the Yoga Pose Template

This is a Jinja2 template that will be used for all yoga pose prompts.
Variables like `{{ character.shay.base_identity }}` reference the YAML wildcard values.

In [ ]:
# Define the yoga pose template
yoga_template = """{{ character.shay.base_identity }}

Full body yoga studio photography. {{ pose_variation_name }}, {{ pose_description }}. Wearing high-waist black athletic leggings with mesh panels and black fitted athletic crop tank. Seamless light-gray studio void with sharp floor shadows, yoga mat clearly visible. {{ style_keyword }}. High-definition 1080p capture, crystal clear sharpness, professional yoga photography lighting. Hair: {{ hairstyle_description }}."""

print("✓ Template defined")
print("\nTemplate preview (first 300 chars):")
print(yoga_template[:300] + "...")

## Load Wildcard Values

Let's inspect what's available in our wildcard system.

In [ ]:
# Load and inspect character data
character_data = builder.loader.load('character')
shay = character_data.get('shay', {})

print("✓ Character: Shay")
print(f"  Hairstyle options: {list(shay.get('hairstyles', {}).keys())}")

# Load pose data
pose_data = builder.loader.load('yoga_poses')
print("\n✓ Available yoga poses:")
for pose_id, pose_info in pose_data.items():
    variations = list(pose_info.get('variations', {}).keys())
    print(f"  {pose_id}: {pose_info.get('english_name')} ({pose_info.get('sanskrit_name')})")
    print(f"    Variations: {variations}")

# Load style data
style_data = builder.loader.load('photographic_styles')
print(f"\n✓ Available photographic styles: {list(style_data.keys())}")

## Define Vajrasana Variations

We'll create 4 variations of the Thunderbolt Pose with different anatomical focuses.

In [ ]:
# Define the 4 Vajrasana variations we want to generate
vajrasana_variations = {
    "base": {
        "id": "VAJ001",
        "pose_id": "vajrasana",
        "variation_id": "base"
    },
    "toe_squat_yin": {
        "id": "VAJ002",
        "pose_id": "vajrasana",
        "variation_id": "toe_squat_yin"
    },
    "side_lunge_prep": {
        "id": "VAJ003",
        "pose_id": "vajrasana",
        "variation_id": "side_lunge_prep"
    },
    "forward_fold": {
        "id": "VAJ004",
        "pose_id": "vajrasana",
        "variation_id": "forward_fold"
    }
}

print(f"✓ Defined {len(vajrasana_variations)} Vajrasana variations")
for name, info in vajrasana_variations.items():
    print(f"  - {name}: {info['id']}")

## Generate Prompts by Rolling Wildcards

Now we'll compose each variation, selecting hairstyles and styles from the wildcards.

In [ ]:
# Generate prompts for each variation
prompts = []

# Loop through Vajrasana variations
for var_name, var_config in vajrasana_variations.items():
    pose_id = var_config["pose_id"]
    variation_id = var_config["variation_id"]
    
    # Get pose data
    pose_info = pose_data[pose_id]
    variation_info = pose_info["variations"][variation_id]
    
    # Roll wildcards for this variation
    hairstyle_key = "high_ponytail"  # Fixed for yoga dynamic poses
    hairstyle_desc = shay["hairstyles"][hairstyle_key]
    
    # Select a style (for demo, use vogue_editorial)
    selected_style = "vogue_editorial"
    style_info = style_data[selected_style]
    style_keyword = style_info["prompt_keywords"][0]
    
    # Prepare selections for template
    selections = {
        "pose_variation_name": variation_info["name"],
        "pose_description": variation_info["description"],
        "hairstyle_description": hairstyle_desc,
        "style_keyword": style_keyword,
        "pose_focus": variation_info["pose_focus"],
        "anatomical_notes": variation_info["anatomical_notes"]
    }
    
    # Prepare metadata
    metadata = {
        "pose_id": pose_id,
        "pose_name": pose_info["english_name"],
        "pose_sanskrit": pose_info["sanskrit_name"],
        "variation_id": variation_id,
        "variation_name": variation_info["name"],
        "pose_focus": variation_info["pose_focus"],
        "hairstyle": hairstyle_key,
        "photographic_style": selected_style,
        "camera_angle": "side_profile",
        "resolution": "1080p"
    }
    
    # Prepare model-specific captions
    captions = {
        "z_image_turbo": f"shay, {variation_info['name'].lower()}, side profile, vogue editorial yoga photography, leggings, ponytail",
        "flux": f"A woman named Shay in {variation_info['name']}, {variation_info['description']}, wearing black yoga leggings, vogue editorial photography, long dark brown hair in ponytail, professional yoga studio",
        "qwen_image": f"Shay, {variation_info['name']}, {variation_info['pose_focus']}, vogue editorial yoga, ponytail hair"
    }
    
    # Compose the prompt
    prompt = builder.compose(
        template=yoga_template,
        selections=selections,
        prompt_type="yoga_pose",
        wildcard_files=["character", "yoga_poses", "photographic_styles"],
        metadata=metadata,
        captions=captions,
        prompt_id=var_config["id"]
    )
    
    prompts.append(prompt)
    print(f"✓ Generated: {prompt.id} - {variation_info['name']}")

print(f"\n✓ Total prompts generated: {len(prompts)}")

## Inspect Generated Prompts

Let's look at one of the generated prompts.

In [ ]:
# Display first prompt
print("EXAMPLE PROMPT OUTPUT:")
print(prompts[0])
print("\n" + "="*80)
print(f"Prompt ID: {prompts[0].id}")
print(f"Type: {prompts[0].prompt_type}")
print(f"Metadata keys: {list(prompts[0].metadata.keys())}")

## Export All Prompts

Write the prompts to files for ComfyUI batch processing.

In [ ]:
# Export to text file (for nanobanana)
output_txt = Path.cwd() / 'data' / 'Prompts' / 'vajrasana_batch_1080p.txt'

with open(output_txt, 'w', encoding='utf-8') as f:
    f.write("=" * 80 + "\n")
    f.write("VAJRASANA VARIATIONS - SHAY CHARACTER FOR NANOBANANA-3\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Total Prompts: {len(prompts)}\n")
    f.write(f"Resolution: 1080p\n")
    f.write(f"Style: Vogue Editorial\n\n")
    
    for prompt in prompts:
        f.write(str(prompt))

print(f"✓ Exported to text: {output_txt}")
print(f"  File size: {output_txt.stat().st_size} bytes")

In [ ]:
# Export to JSON (for programmatic processing)
output_json = Path.cwd() / 'data' / 'Prompts' / 'vajrasana_batch_1080p.json'

json_data = {
    'model': 'nanobanana-3',
    'total_prompts': len(prompts),
    'style': 'vogue_editorial',
    'resolution': '1080p',
    'prompts': [p.to_dict() for p in prompts]
}

with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(json_data, f, indent=2, ensure_ascii=False)

print(f"✓ Exported to JSON: {output_json}")
print(f"  File size: {output_json.stat().st_size} bytes")

## Summary

We've successfully demonstrated the prompt builder system:

In [ ]:
print("SUMMARY")
print("="*80)
print(f"\n✓ Generated {len(prompts)} yoga pose prompts")
print(f"✓ Pose: Vajrasana (Thunderbolt Pose)")
print(f"✓ Variations: base, toe_squat_yin, side_lunge_prep, forward_fold")
print(f"✓ Character: Shay (Polish-American, hazel eyes, dark brown hair)")
print(f"✓ Style: Vogue Editorial (professional studio lighting, Phase One aesthetics)")
print(f"✓ Resolution: 1080p (optimal for batch testing)")
print(f"\nExported files:")
print(f"  - {output_txt.name}")
print(f"  - {output_json.name}")
print(f"\nNext steps:")
print(f"  1. Load .txt or .json file into ComfyUI NanoBanana node")
print(f"  2. Connect 6 Shay reference images")
print(f"  3. Set image_count=4 per prompt (16 total)")
print(f"  4. Execute workflow to generate images")